# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder)without this

/home/bernard/Projects/dic/Week 3


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *
from ingestion import ingest, transform, DB_SRC, PROJECT_ROOT
from Task_3 import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

:: loading settings :: url = jar:file:/home/bernard/Projects/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/bernard/.ivy2.5.2/cache
The jars for the packages stored in: /home/bernard/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7c8bea7d-f1e3-41d6-ab83-5c9eae05ea6d;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#d

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='pipeline_monitor', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='schema_versions', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# double-check new data

In [3]:
for config_file in Path(f"{PROJECT_ROOT}/ingestion_update_configuration").glob("*.json"):
    print(f"Processing config file: {config_file}")
    df, config = ingest(spark, config_file)
    new_df = transform(df, config)

    new_df.printSchema()
    new_df.show(truncate=False)

Processing config file: /home/bernard/Projects/dic/ingestion_update_configuration/air_quality.json


26/09/25 23:23:17 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /home/bernard/Projects/dic/updates/air_quality_update/*.csv.
java.io.FileNotFoundException: File /home/bernard/Projects/dic/updates/air_quality_update/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource

root
 |-- county: string (nullable = true)
 |-- datetime: timestamp (nullable = false)
 |-- measurement: float (nullable = true)
 |-- aqi: float (nullable = true)

+------------+-------------------+-----------+-----+
|county      |datetime           |measurement|aqi  |
+------------+-------------------+-----------+-----+
|Jefferson   |2025-01-07 21:00:00|12.580212  |189.0|
|Cache       |2025-02-07 18:00:00|1.9884071  |441.0|
|Swain       |2025-02-08 02:00:00|2.9130187  |333.0|
|Jefferson   |2025-02-16 07:00:00|10.352773  |434.0|
|Sumter      |2025-02-17 03:00:00|3.9109704  |5.0  |
|Cache       |2025-02-20 22:00:00|19.022097  |441.0|
|Sullivan    |2025-02-22 08:00:00|6.4462867  |400.0|
|Swain       |2025-03-15 10:00:00|10.924814  |333.0|
|Jefferson   |2025-03-16 21:00:00|6.8585715  |189.0|
|Cache       |2025-03-18 16:00:00|11.167275  |441.0|
|Ozaukee     |2025-03-20 12:00:00|10.742693  |481.0|
|Mitchell    |2025-03-21 23:00:00|17.324375  |329.0|
|DeKalb      |2025-03-30 12:00:00|4.64513

26/09/25 23:23:19 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /home/bernard/Projects/dic/updates/taxi_trips_update/*.parquet.
java.io.FileNotFoundException: File /home/bernard/Projects/dic/updates/taxi_trips_update/*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1Batch

root
 |-- pu_datetime: timestamp (nullable = true)
 |-- do_datetime: timestamp (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- trip_distance: float (nullable = true)

+-------------------+-------------------+--------------+--------------+-----------+-------------+
|pu_datetime        |do_datetime        |pu_location_id|do_location_id|fare_amount|trip_distance|
+-------------------+-------------------+--------------+--------------+-----------+-------------+
|2024-01-01 00:20:11|2024-01-01 00:42:53|4             |238           |28.9       |5.88         |
|2024-01-01 00:15:34|2024-01-01 00:28:51|162           |143           |14.2       |2.1          |
|2024-01-01 00:36:26|2024-01-01 01:08:23|148           |244           |47.8       |11.48        |
|2024-01-01 00:44:37|2024-01-01 00:56:29|68            |107           |11.4       |1.01         |
|2024-01-01 00:24:05|2024-01-01 00:3

26/09/25 23:23:20 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /home/bernard/Projects/dic/updates/weather_update/*.csv.
java.io.FileNotFoundException: File /home/bernard/Projects/dic/updates/weather_update/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(Resolve

root
 |-- datetime: timestamp (nullable = false)
 |-- temp: float (nullable = true)
 |-- rhum: integer (nullable = true)
 |-- prcp: float (nullable = true)
 |-- snwd: float (nullable = true)
 |-- wdir: integer (nullable = true)
 |-- wspd: float (nullable = true)
 |-- cldc: integer (nullable = true)
 |-- coco: integer (nullable = true)
 |-- humidity: integer (nullable = true)

+-------------------+----------+----+-----------+----+----+---------+----+----+--------+
|datetime           |temp      |rhum|prcp       |snwd|wdir|wspd     |cldc|coco|humidity|
+-------------------+----------+----+-----------+----+----+---------+----+----+--------+
|2024-12-31 00:00:00|35.6      |45  |2.2771893  |0.0 |25  |36.985504|8   |8   |45      |
|2024-12-31 01:00:00|16.09721  |49  |0.3064257  |0.0 |136 |18.394165|4   |4   |49      |
|2024-12-31 02:00:00|21.07102  |33  |0.793313   |0.0 |260 |22.987251|2   |14  |33      |
|2024-12-31 03:00:00|9.535672  |57  |0.0        |0.0 |40  |12.334884|3   |3   |57      

# Make sure schema_versions have been noted for the initial datasets

In [4]:
# create schema_versions table if not exists
initialize_registry(spark)

for config_file in Path(f"{PROJECT_ROOT}/ingestion_update_configuration").glob("*.json"):
    print(f"Processing config file: {config_file}")
    with open(config_file) as f:
        config = json.load(f)


    print(f"current schema version for {config['name']}: {register_schema_version(spark, config['name'])}")

Processing config file: /home/bernard/Projects/dic/ingestion_update_configuration/air_quality.json


Current Delta version for air_quality: 3
Current schema for air_quality: {"fields":[{"metadata":{},"name":"county","nullable":true,"type":"string"},{"metadata":{},"name":"datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"measurement","nullable":true,"type":"float"},{"metadata":{},"name":"aqi","nullable":true,"type":"float"}],"type":"struct"}


26/09/25 23:23:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

current schema version for air_quality: 1
Processing config file: /home/bernard/Projects/dic/ingestion_update_configuration/taxi_trips.json
Current Delta version for taxi_trips: 3
Current schema for taxi_trips: {"fields":[{"metadata":{},"name":"pu_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"do_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"pu_location_id","nullable":true,"type":"integer"},{"metadata":{},"name":"do_location_id","nullable":true,"type":"integer"},{"metadata":{},"name":"fare_amount","nullable":true,"type":"float"},{"metadata":{},"name":"trip_distance","nullable":true,"type":"float"}],"type":"struct"}
current schema version for taxi_trips: 0
Processing config file: /home/bernard/Projects/dic/ingestion_update_configuration/weather.json
Current Delta version for weather: 2
Current schema for weather: {"fields":[{"metadata":{},"name":"datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"temp","nullable":true,"type"

In [5]:
# also register enriched table and data products
register_schema_version(spark, "integrated_taxi_trips")
register_schema_version(spark, "daily_county_demand")
register_schema_version(spark, "weather_impact_demand")
register_schema_version(spark, "week_day_demand")
register_schema_version(spark, "conuty_trip_flow")

spark.table("schema_versions").show(truncate=False)

Current Delta version for integrated_taxi_trips: 0
Current schema for integrated_taxi_trips: {"fields":[{"metadata":{},"name":"pu_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"do_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"fare_amount","nullable":true,"type":"float"},{"metadata":{},"name":"trip_distance","nullable":true,"type":"float"},{"metadata":{},"name":"pu_zone","nullable":true,"type":"string"},{"metadata":{},"name":"pu_county","nullable":true,"type":"string"},{"metadata":{},"name":"do_zone","nullable":true,"type":"string"},{"metadata":{},"name":"do_county","nullable":true,"type":"string"},{"metadata":{},"name":"measurement","nullable":true,"type":"float"},{"metadata":{},"name":"temp","nullable":true,"type":"float"},{"metadata":{},"name":"rhum","nullable":true,"type":"integer"},{"metadata":{},"name":"prcp","nullable":true,"type":"float"},{"metadata":{},"name":"snwd","nullable":true,"type":"float"},{"metadata":{},"name":"wdir","nullab

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `daily_county_demand` cannot be found. Verify the spelling and correctness of the schema and catalog.
Search path: [`system`.`session`, `spark_catalog`.`default`].
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 0;
'DescribeDeltaHistory [version#1569L, timestamp#1570, userId#1571, userName#1572, operation#1573, operationParameters#1574, job#1575, notebook#1576, clusterId#1577, readVersion#1578L, isolationLevel#1579, isBlindAppend#1580, operationMetrics#1581, userMetadata#1582, engineInfo#1583]
+- 'UnresolvedTable [daily_county_demand], DESCRIBE HISTORY, false


# Ingest new data

In [ ]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

for config_file in Path(f"{PROJECT_ROOT}/ingestion_update_configuration").glob("*.json"):
    print(f"Processing config file: {config_file}")
    ingested_df, config = ingest(spark, config_file)
    transformed_df = transform(ingested_df, config)

    transformed_df.printSchema()
    transformed_df.show(truncate=False)

    with log_step("write_delta"):
        # persist delta table
        from delta.tables import DeltaTable
        from functools import reduce

        update_pipeline_execution(spark, transformed_df, config['name'])



# Check if schema_versions have changed since update

In [ ]:
spark.table("schema_versions").show(truncate=False)

# Check pipeline monitor table

In [ ]:
read_pipeline_monitor(spark).show(truncate=False)

# Table histories
Note: The merge only occurs once since subsequent attempts on the same update data is recognized as duplicates

In [ ]:
# see deltalog history
spark.sql("DESCRIBE HISTORY default.air_quality").show(truncate=False)


In [ ]:
# see deltalog history
spark.sql("DESCRIBE HISTORY default.taxi_trips").show(truncate=False)


In [ ]:
# see deltalog history
spark.sql("DESCRIBE HISTORY default.weather").show(truncate=False)


# Table content

In [ ]:
# show the delta table content
spark.table("default.air_quality").show(truncate=False)

In [ ]:
# show the delta table content
spark.table("default.taxi_trips").show(truncate=False)

In [ ]:
# show the delta table content
spark.table("default.weather").show(truncate=False)